In [18]:
import numpy as np

from bokeh.plotting import figure, output_notebook, show

# prepare some data
N = 4000
x = np.random.random(size=N) * 100
y = np.random.random(size=N) * 100
radii = np.random.random(size=N) * 1.5
colors = [
    "#%02x%02x%02x" % (int(r), int(g), 150) for r, g in zip(50+2*x, 30+2*y)
]

# output to static HTML file (with CDN resources)
output_notebook()
TOOLS = "crosshair,pan,wheel_zoom,box_zoom,reset,box_select,lasso_select"

# create a new plot with the tools above, and explicit ranges
p = figure(tools=TOOLS, x_range=(0, 100), y_range=(0, 100))

# add a circle renderer with vectorized colors and sizes
p.circle(x, y, radius=radii, fill_color=colors, fill_alpha=0.6, line_color=None)

# show the results
show(p)

Loading BokehJS ...

In [28]:
import urllib.request
import json
contents = urllib.request.urlopen("http://nginx:8080/dashboard/api/job/?process=3").read()
api = json.loads(contents)
mergedqa = api['results'][0]['output']
print(mergedqa)

{'EXPID': 3578, 'NIGHT': '20191001', 'TASKS': {'CHECK_CCDs': {'PARAMS': {'CUTPIX': 5, 'MAX_SIGMA': 10, 'PERCENTILES': [68.2, 95.4, 99.7], 'PIXEL_RANGE': 7, 'XWSIGMA_DARK_REF': [1.1, 1.8], 'BIAS_AMP_DARK_REF': [176.0, 179.0, 114.0, 174.0], 'NOISE_AMP_DARK_REF': [3.0, 3.0, 2.5, 2.7], 'XWSIGMA_WARN_RANGE': [-2.0, 2.0], 'BIAS_AMP_WARN_RANGE': [-100.0, 100.0], 'LITFRAC_AMP_DARK_REF': [0.3, 0.3, 0.4, 0.4], 'NOISE_AMP_WARN_RANGE': [-2.0, 2.0], 'XWSIGMA_NORMAL_RANGE': [-1.0, 1.0], 'BIAS_AMP_NORMAL_RANGE': [-80.0, 80.0], 'LITFRAC_AMP_WARN_RANGE': [-0.4, 0.4], 'NOISE_AMP_NORMAL_RANGE': [-1.0, 1.0], 'LITFRAC_AMP_NORMAL_RANGE': [-1.0, 1.0]}, 'METRICS': {'XWSIGMA': [1.1101027514345423, 2.4461044095415647], 'BIAS_AMP': [107.2287109375, 138.9933984375, 152.393427734375, 117.226298828125], 'DATA5SIG': 1, 'DIFF1SIG': 0.035, 'DIFF2SIG': 0.414232900390625, 'NOISE_AMP': [2.1200135249169096, 2.0102221419881476, 2.1020444469110924, 2.218244811326276], 'LITFRAC_AMP': [0.34, 0.35, 0.4, 0.37], 'XWSIGMA_AMP': [

In [32]:
    def plot_amp(self, dz, refexp, name="", font_size="1.2vw", description="", wrg=[]):
        ''' Initializing AMP plot
        '''
        ztext, cbarformat = self.set_amp(dz)
        dx = [0, 1, 0, 1]
        dy = [1, 1, 0, 0]

        cmap_tooltip = """
            <div>

                <div>
                    <span style="font-size: 1vw; font-weight: bold; color: #303030;">AMP: </span>
                    <span style="font-size: 1vw; color: #515151;">@amp_number</span>
                </div>
                <div>
                    <span style="font-size: 1vw; font-weight: bold; color: #303030;">counts: </span>
                    <span style="font-size: 1vw; color: #515151">@ztext</span>
                </div>
                <div>
                    <span style="font-size: 1vw; font-weight: bold; color: #303030;">Reference: </span>
                    <span style="font-size: 1vw; color: #515151;">@ref</span>
                </div>
            </div>
        """.replace("counts:", name.replace("_AMP", "")+":")
        hover = HoverTool(tooltips=cmap_tooltip)

        p = Figure(title=name, tools=[hover],
                   x_range=list([-0.5, 1.5]),
                   y_range=list([-0.5, 1.5]),
                   plot_width=450, plot_height=400)

        p.xaxis.axis_label_text_font_size = font_size
        p.legend.label_text_font_size = font_size
        p.title.text_font_size = font_size

        p.xaxis.axis_label= description

        cmap = get_palette("RdBu_r")
        mapper = LinearColorMapper(palette=cmap, low=wrg[0], high=wrg[1],
            nan_color="darkgrey")

        formatter = PrintfTickFormatter(format=cbarformat)
        color_bar = ColorBar(color_mapper=mapper,  major_label_text_align='left',
                             major_label_text_font_size='10pt', label_standoff=2, location=(0, 0),
                             formatter=formatter, title="(Val-Ref)", title_standoff=15,
                              title_text_baseline="alphabetic")
        p.add_layout(color_bar, 'right')

        p.grid.grid_line_color = None
        p.outline_line_color = None
        p.axis.clear
        p.axis.minor_tick_line_color = None

        p.axis.major_label_text_font_size = '0pt'
        p.yaxis.major_label_text_font_size = '0pt'
        p.xaxis.major_tick_line_color = None
        p.xaxis.minor_tick_line_color = None
        p.yaxis.major_tick_line_color = None
        p.yaxis.minor_tick_line_color = None
        p.yaxis.visible = False
        p.xaxis.visible = True

        zvalid = [x if x > -999 else np.nan for x in dz]
        source = ColumnDataSource(
            data=dict(
                x=dx,
                y=dy,
                z=dz,
                zvalid=zvalid,
                ref=["{:.2f}".format(x) for x in refexp],
                zdiff=np.array(zvalid) - np.array(refexp),
                y_offset1=[i+0.15 for i in dy],
                y_offset2=[i-0.10 for i in dy],
                amp=['AMP %s' % i for i in range(1, 5)],
                amp_number=['%s' % i for i in range(1, 5)],
                ztext=ztext,
            )
        )

        text_props = {
            "source": source,
            "angle": 0,
            "color": "black",
            "text_color": "black",
            "text_align": "center",
            "text_baseline": "middle"}

        p.rect("x", "y", .98, .98, 0, source=source,
               fill_color={'field': 'zdiff', 'transform': mapper}, fill_alpha=0.9)

        p.text(x="x", y="y_offset1", text="amp",
               text_font_size="2vw", **text_props)
        p.text(x="x", y="y_offset2", text="ztext",
               text_font_style="bold", text_font_size="2.5vw", **text_props)

        return p


NameError: name 'plot_amp' is not defined